MIDDLEWARE

Middleware provides a way to more tightly control what happens inside the agent. 

Middleware is useful for the following:
1. Tracking agent behavior with logging, analytics, and debugging.
2. Transforming prompts, tool selection, and output formatting.
3. Adding retries, fallbacks, and early termination logic.
4. Applying rate limits, guardrails, and PII detection.

In [9]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

SUMMARIZATION MIDDLEWARE

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. 

Summarization is useful for the following:
1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

BY NUMBER OF MESSAGES

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

config = {"configurable" : {"thread_id" : "conversation1"}}

questions = [
    "What is 2 + 2?",
    "What is 10 * 5?",
    "What is 100 / 4?",
    "What is 15 - 7?",
    "What is 3 * 3?",
    "What is 4 * 4?"
]

for question in questions:
    response = agent.invoke({"messages" : [HumanMessage(content=question)]},config)
    print(f"Messages : {response}")
    print(f"Number of Messages : {len(response['messages'])}")

Messages : {'messages': [HumanMessage(content='What is 2 + 2?', additional_kwargs={}, response_metadata={}, id='38525c43-a083-479b-8f1d-147157b19252'), AIMessage(content='2\u202f+\u202f2 equals **4**.', additional_kwargs={'reasoning_content': 'The user asks: "What is 2 + 2?" Straightforward. The answer is 4. There\'s no policy conflict. Just answer.'}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 79, 'total_tokens': 129, 'completion_time': 0.057053067, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.00391501, 'prompt_tokens_details': None, 'queue_time': 0.308308851, 'total_time': 0.060968077}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4a35f7bd1b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ec82-1cc1-7a23-a10c-68b12923c001-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 50, 'total_tokens': 129, 

BY NUMBER OF TOKENS / FRACTION

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import tool
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city : str) -> str:
    """Get hotels in the given city"""
    return f"""Hotels available in {city} are:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("tokens",550),     # trigger=("fraction",0.005) ~640 tokens
            keep=("tokens",200)         # keep=("fraction",0.002)    ~256 tokens
        )
    ]
)

config = {"configurable" : {"thread_id" : "conversation2"}}

cities = ["Paris","New York","London","Switzerland","Dubai","Singapore"]

def count_tokens(messages):
    total_chars = sum(len(str(message.content)) for message in messages)
    return total_chars // 4

for city in cities:
    response = agent.invoke({"messages" : [HumanMessage(f"Find hotels in {city}")]},config)
    print(f"Messages : {response}")
    print(f"Number of messages : {len(response['messages'])}")
    print(f"Number of tokens : {count_tokens(response['messages'])}")


Messages : {'messages': [HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e3c4102f-6ea9-47a9-a678-cac10d864359'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_8b71fb6a-d286-46ff-a106-664eb7b81997', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 125, 'total_tokens': 163, 'completion_time': 0.043411681, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.008391451, 'prompt_tokens_details': None, 'queue_time': 0.346599485, 'total_time': 0.051803132}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ec82-37b4-7712-b005-caef3a990836-0', tool_calls=[{'name': '

HUMAN IN THE LOOP MIDDLEWARE

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. 

Human-in-the-loop is useful for the following:

1. High-stakes operations requiring human approval (e.g. database writes, financial transactions).
2. Compliance workflows where human oversight is mandatory.
3. Long-running conversations where human feedback guides the agent.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

def write_mail(body : str) -> str:
    """A function to write the body of an email."""
    return f"The body of the email written as {body}."

def send_mail(mail_id : str, subject : str) -> str:
    """A function to send the drafted email to the given email address with the given subject."""
    return f"The email has been sent to {mail_id} with {subject} as the subject of the drafted email."

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[write_mail,send_mail],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_mail" : False,
                "send_mail" : {
                    "allowed_decisions" : ["approve","reject","edit"]
                }
            }
        )
    ]
)

ACCEPT 

In [17]:
from langgraph.types import Command

config = {"configurable" : {"thread_id" : "test-approve"}}

response = agent.invoke({"messages" : [HumanMessage("Send an email to nupur.maheshwari.2010@gmail.com with subject 'Hello' and body 'How are you?'")]},config)

if "__interrupt__" in response:
    print("INTERRUPTED")
    result = agent.invoke(
        Command(
            resume={
                "decisions" : [
                    {"type" : "approve"}
                ]
            }
        ),
        config
    )
    print(result["messages"][-1].content)

INTERRUPTED
The email has been sent to nupur.maheshwari.2010@gmail.com with "Hello" as the subject.


REJECT

In [24]:
from langgraph.types import Command

config = {"configurable" : {"thread_id" : "test-reject"}}

response = agent.invoke({"messages" : [HumanMessage("Send an email to nupur.maheshwari.2010@gmail.com with subject 'Hello' and body 'How are you?'")]},config)

if "__interrupt__" in response:
    print("INTERRUPTED")
    result = agent.invoke(
        Command(
            resume={
                "decisions" : [
                    {"type" : "reject"}
                ]
            }
        ),
        config
    )
    print(result["messages"][-1].content)

INTERRUPTED
I’m sorry, but I can’t send the email right now. If you’d like me to draft it again or do something else, just let me know!


EDIT

In [25]:
from langgraph.types import Command

config = {"configurable" : {"thread_id" : "test-edit"}}

response = agent.invoke({"messages" : [HumanMessage("Send an email to nupur.maheshwari.2010@gmail.com with subject 'Hello' and body 'How are you?'")]},config)

if "__interrupt__" in response:
    print("INTERRUPTED")
    result = agent.invoke(
        Command(
            resume={
                "decisions" : [
                    {
                        "type" : "edit",
                        "edited_action" : {
                            "name" : "send_mail",
                            "args" : {
                                "mail_id" : "nupur.maheshwari.2010@gmail.com",
                                "subject" : "Hi!"
                            }
                        }
                    }
                ]
            }
        ),
        config
    )
    print(result["messages"][-1].content)

INTERRUPTED
I’m sorry for the mix‑up. The email was sent, but the subject line was “Hi!” instead of the “Hello” you requested. If you’d like me to resend it with the correct subject, just let me know!
